# Ex.No 6 — Implementation of a Calculator using LEX and YACC


## AIM
To write a program to implement a calculator using FLEX and BISON.


## ALGORITHM / PROCEDURE
1. Start the program.
2. In the definitions part of the FLEX file, include the regular definition for a digit.
3. In the rules part of the FLEX file, specify the pattern for a number and its action (return `NUM` with the numeric value in `yylval`) in `cal.l`.
4. In the BISON program, define grammar rules so that arithmetic operations +, -, *, / are evaluated using operator precedence.
5. Display an error if the input does not match the grammar.
6. Provide the input.
7. Verify the output.
8. End the program.

**Procedure**
1. Create `cal.l`; define patterns to identify numbers using regular expressions and return `NUM`, storing the value in `yylval`.
2. Create `cal.y`; define grammar rules for arithmetic expressions with `%left`/`%right` precedence and associativity, evaluating `E + E`, `E - E`, `E * E`, `E / E`.
3. Compile: `flex cal.l` → `bison -d cal.y` → `gcc lex.yy.c cal.tab.c -o calc -lfl`.
4. Run `./calc`, enter arithmetic expressions, and view the result for valid/invalid cases.


## PSEUDOCODE / LOGIC
```
GRAMMAR:
    Statement -> E                       { PRINT "Answer: " + E }
    E -> E '+' E   { $$ = $1 + $3 }
       | E '-' E   { $$ = $1 - $3 }
       | E '*' E   { $$ = $1 * $3 }
       | E '/' E   { $$ = $1 / $3 }
       | NUM       { $$ = NUM }

BEGIN
    READ expression tokens
    CALL yyparse() with precedence: (* /) higher than (+ -)
    EVALUATE E bottom-up using the grammar actions above
    PRINT "Answer: " + result
END
```


## PROGRAM & OUTPUT
The cells below contain the source program (FLEX/BISON/C) and its executed output.


In [79]:
# ============================================
# CALCULATOR USING FLEX AND BISON
# Google Colab - Single Cell
# ============================================

# 1. Install FLEX, BISON and GCC
!apt-get update -qq
!apt-get install -y flex bison gcc -qq

# 2. Create cal.l
with open("cal.l", "w") as f:
    f.write(r'''
%{
#include "cal.tab.h"
#include <stdlib.h>
%}

DIGIT [0-9]+(\.[0-9]+)?

%option noyywrap

%%

{DIGIT} {
    yylval.val = atof(yytext);
    return NUM;
}

[ \t] {
}

\n {
    return '\n';
}

. {
    return yytext[0];
}

%%
''')

# 3. Create cal.y
with open("cal.y", "w") as f:
    f.write(r'''
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%union {
    double val;
}

%token <val> NUM

%type <val> E

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

statement:
    E '\n'
    {
        printf("Answer: %g\n", $1);
    }
    ;

E:
      E '+' E
      {
          $$ = $1 + $3;
      }

    | E '-' E
      {
          $$ = $1 - $3;
      }

    | E '*' E
      {
          $$ = $1 * $3;
      }

    | E '/' E
      {
          if ($3 == 0)
          {
              printf("Error: Division by zero\n");
              $$ = 0;
          }
          else
          {
              $$ = $1 / $3;
          }
      }

    | '(' E ')'
      {
          $$ = $2;
      }

    | '-' E %prec UMINUS
      {
          $$ = -$2;
      }

    | NUM
      {
          $$ = $1;
      }
    ;

%%

int main()
{
    printf("Enter the expression:\n");
    yyparse();
    return 0;
}

int yyerror(char *s)
{
    printf("Invalid expression: %s\n", s);
    return 0;
}
''')

# 4. Remove old generated files
!rm -f cal.tab.c cal.tab.h lex.yy.c calc

# 5. Generate BISON and FLEX files
!bison -d cal.y
!flex cal.l

# 6. Compile
!gcc lex.yy.c cal.tab.c -o calc -lfl

# 7. Give input
with open("input.txt", "w") as f:
    f.write("2+2\n")

# 8. Execute
import subprocess

result = subprocess.run(
    ["./calc"],
    stdin=open("input.txt", "r"),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

Enter the expression:
Answer: 4



## RESULT
Thus the program for implementing a calculator using FLEX and BISON was executed and verified successfully.
